# x AI x

**This is the roadmap I followed to build my own AI model from scratch, that will auto-sort my images based on the images content.**

## Pre-Text:

**I have no previous previous AI Development experience, but I do contain the following skills going into this project.**
**This Notebook will assume that all readers have atleast the same skills going into this.**

```markdown
- Stage-1:
  1. Decide if the image is indeed 'Anime art' or not.
  2. Move the image to the appropriate 'yes' or 'no' folder.

- Stage-2:
  1. Decide if the image is indeed 'Anime art' or not.
    - If the image is NOT, move it to the 'no' folder, then stop here.
    - If the image IS 'Anime art', continue onto step 2.
    
  2. Decide the 'Content Group' of the image, and move the image to that groups folder. Current 'Content Groups' include:
    - ''
    - ''
```

---


```txt
Root*/
|-- bin/
    |-- *(binary_files)
|-- images/   
    |-- raw/
        |-- *(image_files)
    |-- output/
        |-- NO/
            |-- *(image_files)
        |-- YES/
            |--  *(image_files)
    |-- training-sets/
        |-- Aspect-Ratio/
            |-- *(image_files)
        |-- Resolution/
            |-- *(image_files)
        |-- YES/
            |-- *(image_files)
        |-- NO/
            |-- *(image_files)
        |-- Unmatched/
            |-- *(image_files)
```

In [ ]:
# All supported Read & Write image Formats

SUPPORTED = {[
  {"AVIF": [".avif"]},
  {"BMP": [".bmp"]},
  {"DDS": [".dds"]},
  {"DIB": [".dib"]},
  {"EPS": [".eps"]},
  {"GIF": [".gif"]},
  {"ICNS": [".icns"]},
  {"ICO": [".ico"]},
  {"JPEG": [".jfif", ".jpe", ".jpeg", ".jpg"]},
  {"MPO": [".mpo"]},
  {"PCX": [".pcx"]},
  {"PNG": [".png"]},
  {"PBM": [".pbm"]},
  {"PGM": [".pgm"]},
  {"PPM": [".ppm"]},
  {"QOI": [".qoi"]},
  {"SGI": [".sgi"]},
  {"TGA": [".tga"]},
  {"TIFF": [".tif", ".tiff"]},
  {"WebP": [".webp"]},
  {"XBM": [".xbm"]},
  {"BLP": [".blp"]},
  {"IM": [".im"]},
  {"MSP": [".msp"]},
  {"SPIDER": [".spi"]}
]}

valid = [
    ".avif", ".bmp", ".dds", ".dib",
    ".eps", ".gif", ".icns", ".ico",
    ".jfif", ".jpe", ".jpeg", ".jpg",
    ".mpo", ".pcx", ".png", ".pbm",
    ".pgm", ".ppm", ".qoi", ".sgi",
    ".tga", ".tif", ".tiff", ".webp",
    ".xbm", ".blp", ".im", ".msp",
    ".spi"
]

In [ ]:
#!/use/bin/env python3

## Ensures all photos are the same encoding type ##

from pathlib import Path
from PIL import Image
from tqdm import tqdm
import os

TARGET_ENCODE = '.png'
TARGET_FORMAT = 'PNG'
IMAGES_DIR = os.path.join(Path.cwd(), "images", "clear")

valid = [
    ".avif", ".bmp", ".dds", ".dib",
    ".eps", ".gif", ".icns", ".ico",
    ".jfif", ".jpe", ".jpeg", ".jpg",
    ".mpo", ".pcx", ".png", ".pbm",
    ".pgm", ".ppm", ".qoi", ".sgi",
    ".tga", ".tif", ".tiff", ".webp",
    ".xbm", ".blp", ".im", ".msp",
    ".spi"
]

# Loop through all items in the directory
for item in tqdm(Path(IMAGES_DIR).iterdir(), desc="Converting Images", unit="image"):
    # Ensure its a file
    if not item.is_file():
        continue
      
    # Ensure its a valid image file otherwise remove from set
    if item.suffix.lower() not in valid:
        continue
    
    # Skip files already matching target encoding
    if item.suffix.lower() == TARGET_ENCODE:
        continue
  
    # Open, Convert, Save, Delete
    try:
        with Image.open(item) as image:
            image = image.convert("RGB")
            output = os.path.join(IMAGES_DIR, f"{item.stem}{TARGET_ENCODE}")
            image.save(output, format=TARGET_FORMAT)
            if Path(output).is_file():
                item.unlink()
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
#!/usr/bin/env python3

from pathlib import Path
from PIL import Image
from math import gcd
from tqdm import tqdm
import os

## Finds the most common Aspect-Ratio and Resolution

IMAGES_DIR = os.path.join(Path.cwd(), "images", "raw")

def calc_aspr(resolution):
    """1. Calculate and return an Images Aspect Ratio"""
    try:
        divisor = gcd(resolution[0], resolution[1])
        return resolution[0] // divisor, resolution[1] // divisor
    except Exception as e:
        print(f"Error: {e}")
        return None

def loop_raw(target: str|Path) -> dict | None:
    """1. Loop through the IMAGES_DIR and print each images Size and Aspect-Ratio.
    2. Counts how many images share the same Size and the same Aspect-Ratio.
    3. Returns the meturn resultost common Resolution and Aspect-Ratio"""
    sizes = {}
    ratios = {}
    for item in tqdm(Path(IMAGES_DIR).iterdir(), desc="Collecting Image Details", unit="image"):
        if not item.is_file():
            continue
        try:
            with Image.open(item) as image:
                resolution = image.size
                aspr = str(calc_aspr(resolution))
                key = f"{resolution[0]}X{resolution[1]}"

                # Count resolutions
                if key not in sizes:
                    sizes[key] = {"count": 0, "ratio": aspr}
                sizes[key]["count"] += 1

                # Count aspect ratios
                ratios[aspr] = ratios.get(aspr, 0) + 1

        except Exception as e:
            print(f"Error processing {item.name}: {e}")
            continue

    if not sizes or not ratios:
        return None

    # Find the most common resolution and aspect ratio
    most_common_size = max(
        sizes.items(), key=lambda item: item[1]["count"]
    )
    most_common_ratio = max(
        ratios.items(), key=lambda item: item[1]
    )
    result = {
        "resolution": most_common_size[0],
        "resolution_count": most_common_size[1]["count"],
        "aspect_ratio": most_common_ratio[0],
        "aspect_ratio_count": most_common_ratio[1],
    }
    print(f"Most common resolution: {result['resolution']} "
          f"({result['resolution_count']} images)")
    print(f"Most common aspect ratio: {result['aspect_ratio']} "
          f"({result['aspect_ratio_count']} images)")

loop_raw(IMAGES_DIR)

In [ ]:
#!/usr/bin/env python3

## Moves all images of the same Aspect-Ratio into the training-set/Aspect-Ratio

from pathlib import Path
from PIL import Image
from math import gcd
from tqdm import tqdm

## Moves all images from RAW into the shared Aspect-Ratio directory

IMAGES_DIR = os.path.join(Path.cwd(), "images", "raw")
TARGET_RATIO = (2,3)

def calc_aspr(resolution):
    """1. Calculate and return an Images Aspect Ratio"""
    try:
        divisor = gcd(resolution[0], resolution[1])
        return resolution[0] // divisor, resolution[1] // divisor
    except Exception as e:
        print(f"Error: {e}")
        return None


for item in tqdm(Path(IMAGES_DIR).iterdir(), desc="Copying Images", unit="image"):
        if not Path(item).is_file():
            continue
        try:
            with Image.open(item) as image:
                name = item.name
                resolution = image.size
                aspr = calc_aspr(resolution)
                if aspr != TARGET_RATIO:
                    continue
                output = os.path.join(Path.cwd(), "images", "training-sets", "Aspect-Ratio", name)
                image.save(Path(output), "PNG")
        except Exception as e:
            print(f"Error processing {item.name}: {e}")
            continue

In [ ]:
#!/usr/bin/env python3

## Moves all images of the same Resolution into the training-set/Resolution directory

from pathlib import Path
from PIL import Image
from math import gcd
from tqdm import tqdm

## Moves all images from RAW into the shared Resolution folder

IMAGES_DIR = os.path.join(Path.cwd(), "images", "raw")
OUTPUT_DIR = os.path.join(Path.cwd(), "images", "training-sets", "Resolution")
TARGET_RES = (832,1248)

for item in tqdm(Path(IMAGES_DIR).iterdir(), desc="Moving Images", unit="image"):
    try:
        with Image.open(item) as image:
            name = item.name
            resolution = image.size
            if resolution != TARGET_RES:
                continue
            output = os.path.join(OUTPUT_DIR, name)
            image.save(Path(output), "PNG")
    except Exception as e:
        print(f"Error: {e}")

Moving Images: 2image [00:00,  2.19image/s]

(832, 1248) : (1170, 1170)


Moving Images: 3image [00:01,  1.99image/s]

(832, 1248) : (816, 1456)


Moving Images: 5image [00:02,  2.32image/s]

...

Moving Images: 798image [06:54,  2.95image/s]

(832, 1248) : (1170, 1170)


Moving Images: 799image [06:54,  2.63image/s]

(832, 1248) : (1024, 1024)


Moving Images: 800image [06:55,  1.93image/s]


In [ ]:
#!/usr/bin/env python3

## An Example Image Pre-Process for model

from pathlib import Path
from tqdm import tqdm
from PIL import Image
import numpy as np
import os

IMAGES_DIR = os.path.join(Path.cwd(), "images", "raw")

class ImageProcess():
    def __init__(self):
        self._dir = Path.cwd()
        self.img_dir = Path(IMAGES_DIR)

    def proc(path:Path) -> None | np.ndarray:
        """Process a single image and return its normalized flattened vector"""
        try:
            if path.is_file():
                image = Image.open(path).convert("RGB")
                array = np.array(image)
                normalized = array.astype(np.float32) / 255.0
                vector = normalized.flatten()
                print(f"Array: {array}")
                print(f"Normalized: {normalized}")
                print(f"Vector: {vector}")
                return vector
            else:
                raise ValueError(f"Error: path must be an image file")
        except Exception as e:
            print(f"Error: {e}")

worker = ImageProcess
target = os.path.join(IMAGES_DIR, "1.png")
worker.proc(path=Path(target))


In [ ]:
#!/home/rpi/Public/xai/bin/python3

from PIL import Image
from pathlib import Path
import numpy as np
from tqdm import tqdm

# Configuration
IMAGE_SIZE = (832, 1024)  # Pillow uses (width, height)
SUPPORTED_FORMATS = {".png", ".jpg", ".jpeg"}
def model_path(filename: str | Path) -> Path:
    """Normalize a model filename to use the .npz extension."""
    return Path(filename).with_suffix(".npz")

# Model Workflows
def create_model(input_count: int, neuron_count: int, filename: str) -> tuple[np.ndarray, np.ndarray]:
    """Create and save a model with configurable inputs and neurons."""
    if input_count <= 0 or neuron_count <= 0:
        raise ValueError("Input and neuron counts must be positive.")
    W = np.random.randn(neuron_count, input_count) * 0.01
    b = np.zeros(neuron_count)
    path = f"{filename}.npz"
    np.savez(path, weights=W, biases=b)
    print(f"Model parameters saved to {path}")
    print("Weight matrix shape:", W.shape)
    print("Bias vector shape:", b.shape)
    return W, b

def load_model(filename: str | Path) -> tuple[np.ndarray, np.ndarray]:
    """Load model weights and biases into memory."""
    path = model_path(filename)
    with np.load(path) as data:
        W = data["weights"]
        b = data["biases"]
    return W, b

# Helpers
def conv(img: str | Path) -> None:
    """Convert an image to RGB and save it as PNG."""
    img = Path(img)
    output_path = img.with_suffix(".png")
    with Image.open(img) as image:
        image.convert("RGB").save(output_path, "PNG")
    print(f"Converted image saved to {output_path}")

def proc_img(filename: str | Path) -> np.ndarray:
    """Process an image into a normalized, flattened RGB vector."""
    path = Path(filename)
    with Image.open(path) as image:
        image = image.convert("RGB")
        image = image.resize(IMAGE_SIZE)
        pixels = np.array(image, dtype=np.float32)
    vector = pixels.flatten() / 255.0
    return vector

# Debug Prints
def pimg(img: str | Path) -> None:
    """Print an image's details and first ten pixel values."""
    path = Path(img)
    with Image.open(path) as image:
        print("Image:", path)
        print("Size:", image.size)
        print("Mode:", image.mode)
        print("First ten pixel values:", list(image.getdata())[:10])

def pimg_diff(img: str | Path) -> None:
    """Print image details and raw RGB values before and after conversion."""
    path = Path(img)
    with Image.open(path) as image:
        print("Original image:")
        print("Size:", image.size)
        print("Mode:", image.mode)
        print("First ten pixel values:", list(image.getdata())[:10])
        rgb_image = image.convert("RGB")
        print("\nAfter RGB conversion:")
        print("Size:", rgb_image.size)
        print("Mode:", rgb_image.mode)
        print("First ten RGB values:", list(rgb_image.getdata())[:10])

# Activation Function
def sigmoid(z: np.ndarray) -> np.ndarray:
    """Map raw neuron outputs to values between zero and one."""
    z = np.clip(z, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(-z))

# Inference
def run(image: str | Path, model: str | Path) -> np.ndarray:
    """Run one forward pass and return raw neuron output."""
    W, B = load_model(model)
    inputs = proc_img(image)
    if inputs.size != W.shape[1]:
        raise ValueError(
            f"Image produces {inputs.size} inputs, "
            f"but model expects {W.shape[1]}."
        )
    return W @ inputs + B

# Training
def train(model: str | Path, yes_dir: str | Path, no_dir: str | Path, epocs: int, learning_rate: float = 0.001) -> tuple[np.ndarray, np.ndarray]:
    """Train a single-output binary classifier with gradient descent."""
    path = model_path(model)
    yes_dir = Path(yes_dir)
    no_dir = Path(no_dir)
    if epocs <= 0:
        raise ValueError("epocs must be positive.")
    if learning_rate <= 0:
        raise ValueError("Learning rate must be positive.")
    if not yes_dir.is_dir() or not no_dir.is_dir():
        raise NotADirectoryError(
            "Both yes_dir and no_dir must be existing directories."
        )
    W, B = load_model(path)
    if W.ndim != 2 or W.shape[0] != 1 or B.shape != (1,):
        raise ValueError(
            "This trainer requires one output neuron and one bias."
        )
    expected_inputs = IMAGE_SIZE[0] * IMAGE_SIZE[1] * 3
    if W.shape[1] != expected_inputs:
        raise ValueError(
    f"Model expects {W.shape[1]} inputs, but proc_img "
            f"produces {expected_inputs}. Recreate the model "
            "or use matching image dimensions."
        )
    samples = []
    for directory, target in (
        (yes_dir, 1.0),
        (no_dir, 0.0),
    ):
        for image_path in directory.iterdir():
            if (
                image_path.is_file()
                and image_path.suffix.lower() in SUPPORTED_FORMATS
            ):
                samples.append((image_path, target))
    if not samples:
        raise ValueError("No supported images found in the training directories.")

    for epoc in tqdm(range(epocs), desc="Training epocs"):
        np.random.shuffle(samples)
        total_loss = 0.0
        progress = tqdm(
            samples,
            desc=f"epoc {epoc + 1}/{epocs}",
            leave=False,
        )
        for i, (image_path, target) in enumerate(progress):
            inputs = proc_img(image_path)

            # Forward pass.
            z = W @ inputs + B
            prediction = sigmoid(z)

            # Binary cross-entropy loss.
            p = float(np.clip(prediction[0], 1e-7, 1.0 - 1e-7))

            loss = -(
                target * np.log(p)
                + (1.0 - target) * np.log(1.0 - p)
            )
            total_loss += loss

            # Backpropagation for sigmoid + binary cross-entropy.
            dz = prediction - target
            dW = dz.reshape(-1, 1) @ inputs.reshape(1, -1)
            dB = dz

            # Gradient descent.
            W -= learning_rate * dW
            B -= learning_rate * dB

            progress.set_postfix(
                loss=f"{total_loss / (i + 1):.4f}"
            )
        tqdm.write(
            f"epoc {epoc + 1}/{epocs} — "
            f"average loss: {total_loss / len(samples):.4f}"
        )
        
    # Use the same parameter keys as create_model() and load_model().
    np.savez(path, weights=W, biases=B)
    print(f"Trained model saved to {path}")
    return W, B

#create_model(filename="xai", neuron_count=1, input_count=2555904)
#path=f"{Path.cwd()}/images/training-sets/Aspect-Ratio/0.png"
#run(path, model="xai.npz")
path=f"{Path.cwd()}/images/training-sets"
train(model="xai.npz", yes_dir=f"{path}/YES" , no_dir=f"{path}/NO", epocs=5)